# Krea2 on Kaggle (ComfyUI, safetensors)

Deploy ComfyUI (latest master) + Ray Artshoot Krea2 v5 nvfp4 (safetensors, UNETLoader) + Qwen3-VL 4B fp8 encoder + Qwen Image VAE.

Ray Artshoot v5 nvfp4 is 7.2 GB, so the standard UNETLoader route fits: Ray 7.2 GB + encoder 5.24 GB + VAE 0.25 GB = ~12.7 GB, inside both /kaggle/working (20 GB) and T4 VRAM (16 GB). The Ray model reaches Kaggle as the private dataset kitzhang2019/ray-artshoot-krea2-v5-nvfp4 attached to this kernel (the App mounts it in runJob), located by a recursive search under /kaggle/input (the real mount path varies, never hard-code it), then copied into ComfyUI's diffusion_models. This mirrors the Moody Moss Mix V3.3 dataset pattern used by the MOSS notebook.

Sampling: 8 steps, cfg 1, ddim, beta via KSamplerAdvanced (add_noise enable, start 0, end 8, return_with_leftover_noise enable) -- the local Herraw workflow sampler. HerRAW LoRA 0.3 by default. Negative is ConditioningZeroOut (the positive conditioning zeroed out), as in the template.

Note: this App template generates the base image only (no SeedVR upscale in the App path yet); the station/disk template adds SeedVR2 7B to 2048x3072.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/working
!python3 -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
!if [ -d /kaggle/working/ComfyUI ]; then echo comfyui-exists; else git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /kaggle/working/ComfyUI; fi
!cd /kaggle/working/ComfyUI && grep -v -E "^(torch|torchvision|torchaudio)(==|>=|~=|>|<| |$)" requirements.txt > /tmp/comfy_reqs.txt && pip install -q -r /tmp/comfy_reqs.txt
!pip install -q torchsde
!python3 -c "import torchsde; print('torchsde ok')"
!ls /kaggle/working/ComfyUI | head -20

In [ ]:
import os
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
from huggingface_hub import hf_hub_download
base = "/kaggle/working/ComfyUI/models"
#
# LoRA (2026-11-15, user decision: HerRAW 0.3 on by default): HerRAW Sensual v0.2
# (Civitai 2863032, version 3338894), strength 0.3 (local workflow
# Herraw-V01-Krea2-v10.json: LoraLoaderModelOnly, 0.3). HF mirror
# RunningHubAI/rh-herraw-sensual-v0.2-lora hosts the same file.
# patchNotebook clears LORA_NAME when the form toggle is off.
LORA_NAME = "HerRAW_Sensual_v0.2.safetensors"
LORA_REPO = "RunningHubAI/rh-herraw-sensual-v0.2-lora"
LORA_STRENGTH = 0.3
# Krea2 files (2026-11-15: main model synced to Ray, same as station/disk template):
# - Ray Artshoot v5 nvfp4 7.2 GB (Civitai 2574319, version 3378023, file 3318813;
#   reaches Kaggle as the private dataset kitzhang2019/ray-artshoot-krea2-v5-nvfp4,
#   located by recursive search under /kaggle/input, then copied into diffusion_models)
# - encoder qwen3vl_4b_fp8_scaled 5.24 GB (Comfy-Org/Krea-2, text_encoders/)
# - VAE qwen_image_vae 0.25 GB (Comfy-Org/Krea-2, vae/)
# Total ~12.7 GB, fits /kaggle/working 20 GB and T4 16 GB VRAM.
import glob, shutil
RAY_NAME = "Ray-Krea2-Artshoot-v5-turbo-nvfp4.safetensors"
hits = sorted(glob.glob("/kaggle/input/**/" + RAY_NAME, recursive=True))
if not hits:
    raise SystemExit("ray model not found under /kaggle/input: dataset kitzhang2019/ray-artshoot-krea2-v5-nvfp4 is not attached to this kernel")
ray_dst = os.path.join(base, "diffusion_models", RAY_NAME)
os.makedirs(os.path.dirname(ray_dst), exist_ok=True)
shutil.copyfile(hits[0], ray_dst)
print("ray model:", hits[0], "->", ray_dst, round(os.path.getsize(ray_dst) / 1e9, 2), "GB")
for fname in ["text_encoders/qwen3vl_4b_fp8_scaled.safetensors", "vae/qwen_image_vae.safetensors"]:
    p = hf_hub_download(repo_id="Comfy-Org/Krea-2", filename=fname, local_dir=base)
    print(fname, round(os.path.getsize(p) / 1e9, 2), "GB")
if LORA_NAME:
    p = hf_hub_download(repo_id=LORA_REPO, filename=LORA_NAME, local_dir=base + "/loras")
    print("lora", LORA_NAME, round(os.path.getsize(p) / 1e9, 2), "GB")
# Guard: the three files must sit exactly where the loaders look, or fail now.
expected = {
    "diffusion_models": RAY_NAME,
    "text_encoders": "qwen3vl_4b_fp8_scaled.safetensors",
    "vae": "qwen_image_vae.safetensors",
}
for folder, fname in expected.items():
    fp = os.path.join(base, folder, fname)
    print(folder + "/", fname, "exists:", os.path.exists(fp))
    assert os.path.exists(fp), "missing model file: " + fp
print("all models downloaded")

In [ ]:
import json, time, requests, os, subprocess, sys

BASE = "http://127.0.0.1:8188"
LOG = "/kaggle/working/comfy_server.log"

# Start the ComfyUI server as a child of this cell's python process (nohup across cells does not survive Kaggle version runs).
logf = open(LOG, "wb")
proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--disable-auto-launch"],
    cwd="/kaggle/working/ComfyUI",
    stdout=logf, stderr=subprocess.STDOUT,
    start_new_session=True,
)
print("server pid:", proc.pid)

up = False
for i in range(36):
    if proc.poll() is not None:
        print("server exited early, code:", proc.returncode)
        break
    try:
        r = requests.get(BASE + "/system_stats", timeout=5)
        if r.status_code == 200:
            up = True
            print("server up after", (i + 1) * 5, "s")
            break
    except Exception:
        pass
    time.sleep(5)
if not up:
    print("server did not come up; log tail:")
    print(open(LOG, errors="ignore").read()[-3000:])
    proc.terminate()
    raise SystemExit("comfyui server failed to start")

PROMPT = "A photorealistic portrait of a young woman in a sunlit garden, soft morning light, shallow depth of field, 50mm lens"
# Loader: standard UNETLoader (safetensors). Ray Artshoot v5 nvfp4 is a
# safetensors checkpoint (2026-11-15: App synced to the station/disk Ray config).
# LoRA settings are defined in the download cell; default to "no LoRA" if that cell was skipped.
LORA_NAME = globals().get("LORA_NAME", "")
LORA_STRENGTH = globals().get("LORA_STRENGTH", 1.0)

info = requests.get(BASE + "/object_info", timeout=120).json()
have = set(info.keys())
need = ["UNETLoader", "CLIPLoader", "VAELoader", "CLIPTextEncode", "ConditioningZeroOut",
        "EmptyLatentImage", "KSamplerAdvanced", "VAEDecode", "SaveImage"]
if LORA_NAME:
    need.append("LoraLoaderModelOnly")
missing = [n for n in need if n not in have]
print("missing nodes:", missing if missing else "none")
if missing:
    proc.terminate()
    raise SystemExit("missing nodes: " + ", ".join(missing))

def node_inputs(class_type, preferred):
    spec = info[class_type]["input"]
    allowed = set(spec.get("required", {})) | set(spec.get("optional", { }))
    return {k: v for k, v in preferred.items() if k in allowed}

# Print the loader + sampler specs to the log so a wrong input name/value is caught here, not after a long run.
for ct in ["UNETLoader", "CLIPLoader", "KSamplerAdvanced"]:
    print(ct, "spec:", json.dumps(info[ct]["input"])[:800])

# Raw-node version of the official Krea2 Turbo T2I template (subgraph expanded,
# prompt-enhancement branch dropped, prompt sent as-is). Core pipeline:
# UNETLoader (Ray nvfp4 safetensors) -> CLIPLoader (type krea2) -> VAELoader; CLIPTextEncode ->
# ConditioningZeroOut (negative); EmptyLatentImage 1024x1024;
# KSamplerAdvanced 8 steps, cfg 1, ddim, beta, add_noise enable, start 0, end 8,
# return_with_leftover_noise enable (weave fix 2026-11-15, disk template); VAEDecode; SaveImage.
nodes = {}
nodes["1"] = {"class_type": "UNETLoader", "inputs": node_inputs("UNETLoader", {"unet_name": "Ray-Krea2-Artshoot-v5-turbo-nvfp4.safetensors", "weight_dtype": "default"})}
nodes["2"] = {"class_type": "CLIPLoader", "inputs": node_inputs("CLIPLoader", {"clip_name": "qwen3vl_4b_fp8_scaled.safetensors", "type": "krea2", "device": "default"})}
nodes["3"] = {"class_type": "VAELoader", "inputs": node_inputs("VAELoader", {"vae_name": "qwen_image_vae.safetensors"})}
nodes["4"] = {"class_type": "CLIPTextEncode", "inputs": {"text": PROMPT, "clip": ["2", 0]}}
nodes["5"] = {"class_type": "ConditioningZeroOut", "inputs": {"conditioning": ["4", 0]}}
nodes["6"] = {"class_type": "EmptyLatentImage", "inputs": node_inputs("EmptyLatentImage", {"width": 1024, "height": 1024, "batch_size": 1})}
# LoRA (optional): when LORA_NAME is set, patch the model with LoraLoaderModelOnly
# (model-only, no CLIP) between the UNETLoader and KSampler. When empty, the
# model goes straight from the UNETLoader to KSampler, unchanged.
MODEL_OUT = ["1", 0]
if LORA_NAME:
    nodes["20"] = {"class_type": "LoraLoaderModelOnly", "inputs": node_inputs("LoraLoaderModelOnly", {"model": ["1", 0], "lora_name": LORA_NAME, "strength_model": LORA_STRENGTH})}
    MODEL_OUT = ["20", 0]
nodes["7"] = {"class_type": "KSamplerAdvanced", "inputs": node_inputs("KSamplerAdvanced", {"model": MODEL_OUT, "add_noise": "enable", "noise_seed": 42, "steps": 8, "cfg": 1, "sampler_name": "ddim", "scheduler": "beta", "positive": ["4", 0], "negative": ["5", 0], "latent_image": ["6", 0], "start_at_step": 0, "end_at_step": 8, "return_with_leftover_noise": "enable", "denoise": 1.0})}
nodes["8"] = {"class_type": "VAEDecode", "inputs": {"samples": ["7", 0], "vae": ["3", 0]}}
nodes["9"] = {"class_type": "SaveImage", "inputs": {"images": ["8", 0], "filename_prefix": "krea2_test"}}

# Pre-check: every required input of every node must be present, or fail here with the
# full list (do not let the server reject it one input at a time). Plus a combo-value
# check: where the node definition lists the allowed values, the value we send must be
# one of them (a bad value can be silently dropped and only fail at execution -- the Wan
# V5 lesson). NOTE: a combo's allowed values live in cfg[0] as a LIST; a scalar input
# (STRING/INT/FLOAT) has cfg[0] as a type-name string, so it is skipped -- this is the
# exact check the Qwen notebooks use, and the Krea2 V1 failure was a wrong simplified
# version of it that treated the free-text prompt as a combo value.
problems = []
for nid, node in nodes.items():
    spec = info[node["class_type"]]["input"]
    req = spec.get("required", {})
    for name in req:
        if name not in node["inputs"]:
            problems.append(nid + ":" + node["class_type"] + "." + name)
    for name, val in node["inputs"].items():
        cfg = req.get(name) or spec.get("optional", {}).get(name)
        if isinstance(cfg, list) and cfg and isinstance(cfg[0], list) and isinstance(val, (str, int, float)):
            if val not in cfg[0]:
                problems.append(nid + ":" + node["class_type"] + "." + name + " value '" + str(val) + "' not in " + str(cfg[0]))
print("missing required inputs:", problems if problems else "none")
if problems:
    proc.terminate()
    raise SystemExit("missing required inputs: " + ", ".join(problems))

r = requests.post(BASE + "/prompt", json={"prompt": nodes}, timeout=180)
print("submit:", r.status_code, r.text[:400])
r.raise_for_status()
pid = r.json()["prompt_id"]
print("prompt_id:", pid)

out_path = "/kaggle/working/krea2_test.png"
t0 = time.time()
done = False
# NOTE: SaveImage writes into ComfyUI/output/, NOT /kaggle/working/. The image must
# be fetched back through the /view API and written to out_path here -- the Krea2 V2
# failure was exactly this step missing (sampling finished in 120 s and node 9 had
# produced krea2_test_00001_.png, but nothing copied it to out_path, so the assert
# below failed). This is the same download step the Qwen/MOSS notebooks use.
while time.time() - t0 < 3600:
    h = requests.get(BASE + "/history/" + pid, timeout=60).json()
    if pid in h:
        st = h[pid].get("status", {})
        sstr = st.get("status_str")
        if sstr == "success":
            outs = h[pid]["outputs"]
            imgs = outs.get("9", {}).get("images", [])
            print("images:", imgs)
            if imgs:
                im = imgs[0]
                q = {"filename": im["filename"], "subfolder": im.get("subfolder", ""), "type": im.get("type", "output")}
                data = requests.get(BASE + "/view", params=q, timeout=300).content
                open(out_path, "wb").write(data)
                print("saved", out_path, len(data), "bytes")
                done = True
        break
    time.sleep(30)
print("done:", done, "elapsed:", round(time.time() - t0, 1), "s")
proc.terminate()
print("server stopped")
print("exists:", os.path.exists(out_path), "bytes:", os.path.getsize(out_path) if os.path.exists(out_path) else 0)
assert os.path.exists(out_path), "no output image: " + out_path

In [ ]:
from IPython.display import Image
p = "/kaggle/working/krea2_test.png"
print(p)
display(Image(filename=p))